# 🟦 01 - Setup

## 🔹 Paths and Configuration

Import libraries

In [0]:
from datetime import datetime
import requests
import pandas as pd
import io
import zipfile
from pyspark.sql.functions import lit, current_timestamp
from pathlib import Path
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import hashlib
from urllib.parse import urlparse
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import shutil




Create volumes

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.raw_files")
print("Done")

Creates text variables for the path 

In [0]:

catalog = "workspace"
schema = "default"
volume = "raw_files"

RAW_BASE = f"/Volumes/{catalog}/{schema}/{volume}/fuel_dashboard/raw"
#run_date = datetime.now().strftime("%Y-%m-%d_%H-%M-%S") 
run_date = datetime.today().strftime("%Y-%m-%d")
TARGETS = {
    "anp_refinery": f"{RAW_BASE}/anp_refinery",
    "anp_imports": f"{RAW_BASE}/anp_imports",
    "brent": f"{RAW_BASE}/brent",
    "anp_prices": f"{RAW_BASE}/anp_prices",
}

print("RAW_BASE:", RAW_BASE)
print("run_date:", run_date)


Check if the volumes are there

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/raw_files"))

# 🟦 02 - Refinery Data (ANP)

- Download data from ANP
- Read with pandas
- Convert to Spark
- Save CSV 

In [0]:
refinery_url = "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/pppd/producao-derivados-petroleo-por-refinaria-m3-1990-2025.csv"
refinery_target = TARGETS["anp_refinery"]
refinery_output_path = f"{refinery_target}/run_date={run_date}"

headers = {"User-Agent": "Mozilla/5.0"}
response_refinery = requests.get(refinery_url, headers=headers, timeout=120)
response_refinery.raise_for_status()

refinery_pd_df = pd.read_csv(
    io.BytesIO(response_refinery.content),
    sep=";",
    encoding="utf-8-sig"
)

refinery_spark_df = spark.createDataFrame(refinery_pd_df)

print("Refinery rows:", refinery_spark_df.count())
display(refinery_spark_df.limit(5))

refinery_spark_df.write.mode("overwrite").option("header", True).csv(refinery_output_path)

print("Saved successfully")
print("Output path:", refinery_output_path)
display(dbutils.fs.ls(refinery_output_path))

# 🟦 03 - Imports Data (ANP)

- Download data from ANP
- Read with pandas
- Convert to Spark
- Save CSV 


In [0]:
imports_url = "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/ie/derivados/importacoes-exportacoes-derivados-2000-2025.csv"
imports_target = TARGETS["anp_imports"]
imports_output_path = f"{imports_target}/run_date={run_date}"

headers = {"User-Agent": "Mozilla/5.0"}
response_imports = requests.get(imports_url, headers=headers, timeout=120)
response_imports.raise_for_status()

imports_pd_df = pd.read_csv(
    io.BytesIO(response_imports.content),
    sep=";",
    encoding="utf-8-sig"
)

imports_spark_df = spark.createDataFrame(imports_pd_df)

print("Imports rows:", imports_spark_df.count())
display(imports_spark_df.limit(5))

imports_spark_df.write.mode("overwrite").option("header", True).csv(imports_output_path)

print("Saved successfully")
print("Output path:", imports_output_path)
display(dbutils.fs.ls(imports_output_path))

# 🟦 04 - Brent Data

- Download Brent daily data from IEA
- Read with pandas
- Convert to Spark
- Save CSV 


In [0]:
brent_url = "https://raw.githubusercontent.com/datasets/oil-prices/master/data/brent-daily.csv"
brent_target = TARGETS["brent"]
brent_output_path = f"{brent_target}/run_date={run_date}"

headers = {"User-Agent": "Mozilla/5.0"}
response_brent = requests.get(brent_url, headers=headers, timeout=120)
response_brent.raise_for_status()

brent_pd_df = pd.read_csv(io.BytesIO(response_brent.content))

print("Brent rows:", len(brent_pd_df))
print(brent_pd_df.columns.tolist())

brent_spark_df = spark.createDataFrame(brent_pd_df)

print("Spark DataFrame created")
print("Row count:", brent_spark_df.count())
display(brent_spark_df.limit(5))

brent_spark_df.write.mode("overwrite").option("header", True).csv(brent_output_path)

print("Saved successfully")
print("Output path:", brent_output_path)
display(dbutils.fs.ls(brent_output_path))

# 🟦 05 - Fuel Prices (ANP)

-Getting url for each file in ANP Data


In [0]:
def url_exists(url: str, timeout: int = 15) -> bool:
    try:
        r = requests.head(url, allow_redirects=True, timeout=timeout)
        if r.status_code == 200:
            return True
        if r.status_code in (403, 405):
            r = requests.get(url, stream=True, allow_redirects=True, timeout=timeout)
            return r.status_code == 200
        return False
    except requests.RequestException:
        return False


def first_existing_url(candidates):
    for url in candidates:
        if url_exists(url):
            return url
    return None


def generate_anp_urls(validate=True):
    base_url = "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/shpc"
    urls = []

    for year in range(2004, 2027):
        for semester in ("01", "02"):
            ca_candidates = [
                f"{base_url}/dsas/ca/ca-{year}-{semester}.zip",
                f"{base_url}/dsas/ca/ca-{year}-{semester}.csv",
            ]
            glp_candidates = [
                f"{base_url}/dsas/glp/glp-{year}-{semester}.csv",
                f"{base_url}/dsas/glp/glp-{year}-{semester}.zip",
            ]

            if validate:
                ca_url = first_existing_url(ca_candidates)
                glp_url = first_existing_url(glp_candidates)
                if ca_url:
                    urls.append(ca_url)
                if glp_url:
                    urls.append(glp_url)

    for year in range(2023, 2027):
        for month in range(1, 13):
            if year == 2026 and month > 2:
                break

            mm = str(month).zfill(2)

            diesel_candidates = [
                f"{base_url}/dsan/{year}/{mm}-dados-abertos-precos-diesel-gnv.csv",
                f"{base_url}/dsan/{year}/{mm}-dados-abertos-preco-diesel-gnv.csv",
            ]

            gasolina_etanol_candidates = [
                f"{base_url}/dsan/{year}/{mm}-dados-abertos-precos-gasolina-etanol.csv",
                f"{base_url}/dsan/{year}/{mm}-dados-abertos-preco-gasolina-etanol.csv",
                f"{base_url}/dsan/{year}/{mm}-cados-abertos-precos-gasolina-etanol.csv",
                f"{base_url}/dsan/{year}/{mm}-cados-abertos-preco-gasolina-etanol.csv",
            ]

            glp_monthly_candidates = [
                f"{base_url}/dsan/{year}/{mm}-dados-abertos-precos-glp.csv",
                f"{base_url}/dsan/{year}/{mm}-dados-abertos-preco-glp.csv",
            ]

            for candidates in [diesel_candidates, gasolina_etanol_candidates, glp_monthly_candidates]:
                found = first_existing_url(candidates) if validate else candidates[0]
                if found:
                    urls.append(found)

    last_4_weeks_candidates = [
        [f"{base_url}/qus/ultimas-4-semanas-diesel-gnv.csv"],
        [f"{base_url}/qus/ultimas-4-semanas-gasolina-etanol.csv"],
        [f"{base_url}/qus/ultimas-4-semanas-glp.csv"],
    ]

    for candidates in last_4_weeks_candidates:
        found = first_existing_url(candidates) if validate else candidates[0]
        if found:
            urls.append(found)

    return sorted(set(urls))

In [0]:
urls = generate_anp_urls(validate=True)

print(len(urls))
print(urls[:5])  # preview first 5

URL Map & Download of files

In [0]:
def generate_safe_filename(url: str) -> str:
    original_name = urlparse(url).path.split("/")[-1]
    url_hash = hashlib.md5(url.encode("utf-8")).hexdigest()[:8]
    return f"{url_hash}_{original_name}"

In [0]:

def download_and_land_file(url, target_volume, run_date, overwrite=False, retries=5):
    headers = {"User-Agent": "Mozilla/5.0"}
    raw_target = f"{target_volume}/run_date={run_date}"
    Path(raw_target).mkdir(parents=True, exist_ok=True)

    original_filename = url.split("/")[-1]
    safe_filename = generate_safe_filename(url)

    if url.endswith(".zip"):
        final_name = safe_filename.replace(".zip", ".csv")
    else:
        final_name = safe_filename

    dest_path = f"{raw_target}/{final_name}"
    temp_path = f"{dest_path}.part"

    def looks_like_html(text_sample: str) -> bool:
        sample = (text_sample or "").strip().lower()
        return (
            sample.startswith("<!doctype html")
            or sample.startswith("<html")
            or "<html" in sample[:500]
            or "<body" in sample[:500]
        )

    def validate_saved_csv(path: str) -> int:
        if not os.path.exists(path):
            raise ValueError("File was not created")

        file_size = os.path.getsize(path)
        if file_size == 0:
            raise ValueError("Downloaded file is empty")

        with open(path, "rb") as f:
            head = f.read(4096)

        head_text = head.decode("utf-8", errors="ignore")

        if looks_like_html(head_text):
            raise ValueError("Downloaded HTML instead of CSV")

        with open(path, "r", encoding="utf-8", errors="ignore", newline="") as f:
            sample = f.read(4096)
            if not sample.strip():
                raise ValueError("CSV is blank")
            try:
                csv.Sniffer().sniff(sample)
            except Exception:
                pass

        return file_size

    if (not overwrite) and Path(dest_path).exists() and os.path.getsize(dest_path) > 0:
        return {
            "source_url": url,
            "original_file_name": original_filename,
            "saved_file_name": final_name,
            "raw_path": dest_path,
            "status": "skipped_existing",
            "error_message": None,
            "file_size_bytes": os.path.getsize(dest_path),
        }

    last_error = None

    for attempt in range(1, retries + 1):
        try:
            if os.path.exists(temp_path):
                os.remove(temp_path)

            response = requests.get(
                url,
                headers=headers,
                stream=True,
                timeout=(30, 300),
                allow_redirects=True
            )
            response.raise_for_status()

            content_type = (response.headers.get("Content-Type") or "").lower()

            if url.endswith(".zip"):
                content = io.BytesIO()

                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        content.write(chunk)

                if content.tell() == 0:
                    raise ValueError("ZIP response is empty")

                content.seek(0)

                with zipfile.ZipFile(content) as z:
                    csv_names = [n for n in z.namelist() if n.lower().endswith(".csv")]
                    if not csv_names:
                        raise ValueError(f"No CSV found inside ZIP: {url}")

                    csv_name = csv_names[0]

                    with z.open(csv_name) as f_in, open(temp_path, "wb") as f_out:
                        while True:
                            chunk = f_in.read(1024 * 1024)
                            if not chunk:
                                break
                            f_out.write(chunk)

            else:
                sample_chunks = []
                sample_size = 0
                max_sample = 4096

                iterator = response.iter_content(chunk_size=1024)

                for chunk in iterator:
                    if chunk:
                        sample_chunks.append(chunk)
                        sample_size += len(chunk)
                        if sample_size >= max_sample:
                            break

                sample_bytes = b"".join(sample_chunks)
                sample_text = sample_bytes.decode("utf-8", errors="ignore")

                if looks_like_html(sample_text):
                    raise ValueError(f"Downloaded HTML instead of CSV. Content-Type={content_type}")

                with open(temp_path, "wb") as f:
                    if sample_bytes:
                        f.write(sample_bytes)

                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)

            file_size = validate_saved_csv(temp_path)

            os.replace(temp_path, dest_path)

            return {
                "source_url": url,
                "original_file_name": original_filename,
                "saved_file_name": final_name,
                "raw_path": dest_path,
                "status": "success",
                "error_message": None,
                "file_size_bytes": file_size,
            }

        except Exception as e:
            last_error = e
            print(f"⚠️ Attempt {attempt}/{retries} failed for {original_filename}: {e}")

            if os.path.exists(temp_path):
                try:
                    os.remove(temp_path)
                except Exception:
                    pass

            time.sleep(min(attempt * 10, 60))

    if os.path.exists(temp_path):
        try:
            os.remove(temp_path)
        except Exception:
            pass

    return {
        "source_url": url,
        "original_file_name": original_filename,
        "saved_file_name": final_name,
        "raw_path": None,
        "status": "failed",
        "error_message": str(last_error),
        "file_size_bytes": None,
    }

In [0]:

prices_target = TARGETS["anp_prices"]
all_links = generate_anp_urls()

max_workers = 3   # safer than 4 for unstable large downloads
prices_ingest_log = []

print(f"Total files to process: {len(all_links)}")


def run_download_batch(urls, overwrite=False, pass_name="pass_1"):
    batch_log = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_url = {
            executor.submit(
                download_and_land_file,
                url,
                prices_target,
                run_date,
                overwrite
            ): url
            for url in urls
        }

        for i, future in enumerate(as_completed(future_to_url), start=1):
            url = future_to_url[future]

            try:
                result = future.result()
            except Exception as e:
                result = {
                    "source_url": url,
                    "original_file_name": url.split("/")[-1],
                    "saved_file_name": generate_safe_filename(url),
                    "raw_path": None,
                    "status": "failed",
                    "error_message": f"Unhandled future exception: {str(e)}",
                    "file_size_bytes": None,
                }

            row = {
                "source_name": "anp_prices",
                "run_date": run_date,
                "pass_name": pass_name,
                **result
            }
            batch_log.append(row)

            print(
                f"[{i}/{len(urls)}] "
                f"{row['status']} - "
                f"{row['saved_file_name']}"
            )

    return batch_log


# PASS 1
pass_1_log = run_download_batch(all_links, overwrite=False, pass_name="pass_1")
prices_ingest_log.extend(pass_1_log)

# identify failed URLs from pass 1
failed_urls = [
    row["source_url"]
    for row in pass_1_log
    if row["status"] == "failed"
]

print(f"\nPass 1 complete. Failed files: {len(failed_urls)}")

# PASS 2 - retry only failed files
if failed_urls:
    print("\nRetrying failed files in pass 2...\n")
    pass_2_log = run_download_batch(failed_urls, overwrite=True, pass_name="pass_2")
    prices_ingest_log.extend(pass_2_log)

prices_log_df = pd.DataFrame(prices_ingest_log)

# final view
display(prices_log_df)

# summary
status_counts = prices_log_df.groupby(["pass_name", "status"]).size().reset_index(name="count")
display(status_counts)

final_latest = (
    prices_log_df
    .sort_values(["source_url", "pass_name"])
    .groupby("source_url", as_index=False)
    .tail(1)
)

print("\nFinal status summary:")
print(final_latest["status"].value_counts())

failed_final = final_latest[final_latest["status"] == "failed"]
print(f"\nFinal failed files: {len(failed_final)}")


In [0]:
summary_prices = (
    prices_log_df.groupby("status", dropna=False)
    .size()
    .reset_index(name="count")
)

display(summary_prices)

In [0]:
display(prices_log_df['error_message'].unique())

- Validate Raw folder

In [0]:
BASE_RAW = Path("/Volumes/workspace/default/raw_files/fuel_dashboard/raw")

SOURCE_FOLDERS = {
    "anp_refinery": BASE_RAW / "anp_refinery",
    "anp_imports": BASE_RAW / "anp_imports",
    "brent": BASE_RAW / "brent",
    "anp_prices": BASE_RAW / "anp_prices",
}

def validate_raw_folder(folder: Path, sample_csvs: int = 3):
    results = {
        "folder": str(folder),
        "exists": folder.exists(),
        "total_files": 0,
        "csv_files": 0,
        "zip_files": 0,
        "other_files": 0,
        "empty_files": 0,
        "bad_zips": 0,
        "readable_csv_samples": 0,
        "csv_sample_errors": 0,
    }

    if not folder.exists():
        return results

    all_files = [f for f in folder.rglob("*") if f.is_file()]
    csv_files = [f for f in folder.rglob("*") if f.is_file() and f.suffix.lower() == ".csv"]
    zip_files = [f for f in folder.rglob("*") if f.is_file() and f.suffix.lower() == ".zip"]

    results["total_files"] = len(all_files)
    results["csv_files"] = len(csv_files)
    results["zip_files"] = len(zip_files)
    results["other_files"] = len(all_files) - len(csv_files) - len(zip_files)

    for f in all_files:
        if os.path.getsize(f) == 0:
            results["empty_files"] += 1

    for z in zip_files:
        try:
            with zipfile.ZipFile(z, "r") as zf:
                zf.testzip()
        except Exception:
            results["bad_zips"] += 1

    attempts = [
        {"sep": ";", "encoding": "latin1"},
        {"sep": ",", "encoding": "utf-8"},
        {"sep": ",", "encoding": "latin1"},
        {"sep": ";", "encoding": "utf-8"},
    ]

    for f in csv_files[:sample_csvs]:
        read_ok = False
        for opts in attempts:
            try:
                pd.read_csv(f, nrows=5, **opts)
                results["readable_csv_samples"] += 1
                read_ok = True
                break
            except Exception:
                pass

        if not read_ok:
            results["csv_sample_errors"] += 1

    return results

In [0]:
summary = []

for name, folder in SOURCE_FOLDERS.items():
    result = validate_raw_folder(folder, sample_csvs=5)
    result["source_name"] = name
    summary.append(result)

summary_df = pd.DataFrame(summary)
display(summary_df)

- Check for duplicated files

In [0]:

base_path = "/Volumes/workspace/default/raw_files/fuel_dashboard/raw/anp_refinery"

for folder in os.listdir(base_path):
    print(folder)

In [0]:
latest_path = "/Volumes/workspace/default/raw_files/fuel_dashboard/raw/anp_refinery/run_date=2026-03-25"

files = os.listdir(latest_path)

print("Total files:", len(files))
print("CSV files:", len([f for f in files if f.endswith(".csv")]))

In [0]:

latest_path = "/Volumes/workspace/default/raw_files/fuel_dashboard/raw/anp_refinery/run_date=2026-03-25"

files = os.listdir(latest_path)

non_csv = [f for f in files if not f.lower().endswith(".csv")]
print("Non-CSV files:")
for f in non_csv:
    full = os.path.join(latest_path, f)
    size = os.path.getsize(full)
    print(f, "| size:", size)

In [0]:

latest_path = "/Volumes/workspace/default/raw_files/fuel_dashboard/raw/anp_refinery/run_date=2026-03-25"

csv_files = [f for f in os.listdir(latest_path) if f.lower().endswith(".csv")]

for f in csv_files:
    print(f)

In [0]:
def count_duplicate_filenames(folder: Path):
    files = [f.name for f in folder.rglob("*") if f.is_file()]
    if not files:
        return 0
    return pd.Series(files).duplicated().sum()

for name, folder in SOURCE_FOLDERS.items():
    print(name, "duplicate filenames:", count_duplicate_filenames(folder))

In [0]:


def summarize_raw_folder(folder_path, source_name):
    if not os.path.exists(folder_path):
        return {
            "folder": folder_path,
            "exists": False,
            "total_files": 0,
            "csv_files": 0,
            "zip_files": 0,
            "other_files": 0,
            "empty_files": 0,
            "bad_zips": 0,
            "readable_csv_samples": 0,
            "csv_sample_errors": 0,
            "spark_metadata_files": 0,
            "source_name": source_name,
        }

    all_entries = os.listdir(folder_path)

    files = [
        f for f in all_entries
        if os.path.isfile(os.path.join(folder_path, f))
    ]

    # arquivos auxiliares do Spark / sistema
    spark_metadata = [
        f for f in files
        if f.startswith("_") or f.startswith(".")
    ]

    # arquivos de negócio
    business_files = [
        f for f in files
        if not (f.startswith("_") or f.startswith("."))
    ]

    csv_files = [f for f in business_files if f.lower().endswith(".csv")]
    zip_files = [f for f in business_files if f.lower().endswith(".zip")]
    other_files = [
        f for f in business_files
        if not f.lower().endswith(".csv") and not f.lower().endswith(".zip")
    ]

    empty_files = [
        f for f in business_files
        if os.path.getsize(os.path.join(folder_path, f)) == 0
    ]

    bad_zips = 0
    for f in zip_files:
        full = os.path.join(folder_path, f)
        try:
            import zipfile
            with zipfile.ZipFile(full, "r") as z:
                bad_file = z.testzip()
                if bad_file is not None:
                    bad_zips += 1
        except Exception:
            bad_zips += 1

    readable_csv_samples = 0
    csv_sample_errors = 0

    for f in csv_files[:5]:
        full = os.path.join(folder_path, f)
        try:
            with open(full, "r", encoding="utf-8", errors="ignore") as file:
                sample = file.read(2048)
                if sample.strip():
                    readable_csv_samples += 1
                else:
                    csv_sample_errors += 1
        except Exception:
            csv_sample_errors += 1

    return {
        "folder": folder_path,
        "exists": True,
        "total_files": len(business_files),
        "csv_files": len(csv_files),
        "zip_files": len(zip_files),
        "other_files": len(other_files),
        "empty_files": len(empty_files),
        "bad_zips": bad_zips,
        "readable_csv_samples": readable_csv_samples,
        "csv_sample_errors": csv_sample_errors,
        "spark_metadata_files": len(spark_metadata),
        "source_name": source_name,
    }

In [0]:
summary_rows = []

summary_rows.append(
    summarize_raw_folder(
        "/Volumes/workspace/default/raw_files/fuel_dashboard/raw/anp_refinery/run_date=2026-03-25",
        "anp_refinery"
    )
)

summary_rows.append(
    summarize_raw_folder(
        "/Volumes/workspace/default/raw_files/fuel_dashboard/raw/anp_imports/run_date=2026-03-25",
        "anp_imports"
    )
)

summary_rows.append(
    summarize_raw_folder(
        "/Volumes/workspace/default/raw_files/fuel_dashboard/raw/brent/run_date=2026-03-25",
        "brent"
    )
)

summary_rows.append(
    summarize_raw_folder(
        "/Volumes/workspace/default/raw_files/fuel_dashboard/raw/anp_prices/run_date=2026-03-25",
        "anp_prices"
    )
)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)